# 04b · Mapas: la caída se ve desde el espacio
> **Workshop Tec de Monterrey · De los datos a los agentes** — Abarrotera del Centro S.A. (empresa ficticia, datos sintéticos).

**Pregunta de negocio:** ¿la caída de Bebidas es un problema de *todo el país* o tiene una geografía? Si tiene geografía, ¿qué la explica?

Usamos la capa `TIENDAS_GEO` (02) a través de la Semantic View (04): 20,000 tiendas en 93 ciudades reales con latitud, longitud y hexágono H3. Como todo pasa por la Row Access Policy, **cada persona ve solo los puntos de su región**.

| Mapa | Qué muestra | Tipo |
|---|---|---|
| 0 | Los 5 CEDIS | Puntos |
| 1 | Cambio de venta de Bebidas sep→oct por ciudad | Burbujas |
| 2 | Densidad de venta de octubre | Hexágonos H3 |
| 3 | La frontera Bajío–Centro | Hexágonos H3 |
| 4 | El mismo mapa como GERENTE_NORTE | Gobierno |

Patrones tomados del sample **Overture Maps Semantic Layer** (Snowflake-Labs): agrupar coordenadas siempre por ID, H3 como texto y geometría simplificada solo para dibujar.

In [ ]:
%%sql -r ctx_base_esquema
-- Contexto de ejecución (primera celda, requerido en runtimes < 2.4)
USE ROLE TEC_ADMIN;
USE WAREHOUSE TEC_WH;
USE DATABASE TEC_DEMO;
USE SCHEMA TEC_DEMO.ANALITICA;

In [ ]:
# Base de los mapas: Snowsight no carga mapas base de internet, así que dibujamos México nosotros.
# go.Scatter con ejes longitud/latitud (escala 1:1), contorno simplificado embebido (~80 vértices, ilustrativo) y mar azul.
import json
import pandas as pd
import plotly.graph_objects as go

MEXICO = [(-117.12,32.53),(-114.72,32.72),(-114.81,32.49),(-111.07,31.33),(-108.21,31.33),(-108.21,31.78),(-106.51,31.75),
  (-104.92,30.60),(-104.40,29.57),(-103.10,28.97),(-102.40,29.77),(-101.40,29.77),(-100.50,28.30),(-99.50,27.50),(-99.10,26.40),
  (-97.50,25.90),(-97.40,25.00),(-97.70,24.00),(-97.80,22.30),(-97.20,20.60),(-96.10,19.20),(-95.20,18.70),(-94.40,18.15),
  (-92.60,18.60),(-91.40,18.60),(-90.70,19.30),(-90.40,20.90),(-90.00,21.20),(-88.40,21.60),(-87.00,21.50),(-86.80,20.60),
  (-87.50,19.50),(-87.80,18.30),(-88.30,18.50),(-89.15,17.95),(-90.98,17.82),(-91.00,17.25),(-91.40,16.07),(-92.20,15.25),
  (-92.24,14.53),(-93.40,15.60),(-94.60,16.20),(-95.20,16.17),(-96.50,15.65),(-98.00,16.10),(-99.90,16.85),(-101.60,17.60),
  (-102.40,17.95),(-103.50,18.30),(-104.30,19.10),(-105.30,19.90),(-105.40,20.55),(-105.20,21.50),(-105.70,22.60),(-106.40,23.20),
  (-107.40,24.40),(-108.60,25.40),(-109.40,26.10),(-109.90,27.00),(-110.60,27.90),(-111.30,28.40),(-112.20,29.00),(-112.80,30.10),
  (-113.10,31.20),(-114.80,31.80),(-114.70,30.90),(-114.00,29.60),(-113.30,28.80),(-112.50,27.80),(-111.60,26.60),(-110.70,24.90),
  (-110.30,24.20),(-109.40,23.40),(-109.90,22.90),(-110.90,24.00),(-112.10,24.80),(-112.10,25.90),(-113.20,26.70),(-114.10,27.60),
  (-114.20,28.00),(-115.80,29.80),(-116.30,30.50),(-116.60,31.80),(-117.12,32.53)]

def mapa_base(titulo, lon=(-118, -86), lat=(14, 33), height=600, mares=True):
    """Figura con el mar, el territorio nacional y ejes lon/lat a escala 1:1."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[p[0] for p in MEXICO], y=[p[1] for p in MEXICO], fill='toself', mode='lines',
                             fillcolor='#f6f1e7', line={'color': '#a6a197', 'width': 1}, hoverinfo='skip', showlegend=False))
    if mares:
        fig.add_annotation(x=-93.5, y=24.5, text='Golfo de México', showarrow=False, font={'color': '#5d7fa3', 'size': 12})
        fig.add_annotation(x=-110.5, y=18.0, text='Océano Pacífico', showarrow=False, font={'color': '#5d7fa3', 'size': 12})
    fig.update_xaxes(range=list(lon), showgrid=False, zeroline=False, visible=False)
    fig.update_yaxes(range=list(lat), showgrid=False, zeroline=False, visible=False, scaleanchor='x', scaleratio=1)
    fig.update_layout(title=titulo, height=height, plot_bgcolor='#dcebf7', margin={'l': 10, 'r': 10, 't': 50, 'b': 10},
                      legend={'x': 0.01, 'y': 0.02, 'bgcolor': 'rgba(255,255,255,0.85)'})
    return fig

def a_pandas(df, numericas=()):
    """Resultado de una celda SQL → pandas con columnas en minúsculas y numéricas como float (NUMBER llega como Decimal)."""
    d = df.to_pandas() if hasattr(df, 'to_pandas') else df.copy()
    d.columns = [c.lower() for c in d.columns]
    for c in numericas:
        d[c] = d[c].astype(float)
    return d

def centro_hex(geojson):
    """Centro aproximado de un hexágono H3 a partir de su polígono GeoJSON."""
    pts = json.loads(geojson)['coordinates'][0][:-1]
    return sum(p[0] for p in pts) / len(pts), sum(p[1] for p in pts) / len(pts)

print('Base de mapas lista')

## Mapa 0 · Calentamiento: los CEDIS
Si tu cuenta tiene habilitados los **mapas nativos de Snowsight** (vista previa privada), en el resultado de esta celda elige la visualización **Map** y liga `LATITUD` / `LONGITUD`. Si no aparece la opción, no pasa nada: las celdas de Python dibujan todos los mapas con plotly sobre un contorno de México embebido (sin depender de mapas base de internet).

In [ ]:
%%sql -r cedis
SELECT cedis_id, nombre, region, latitud, longitud FROM TEC_DEMO.RAW.CEDIS ORDER BY cedis_id;

## Mapa 1 · ¿Dónde cayó Bebidas? (sep → oct 2026, por ciudad)
Tamaño = número de tiendas; color = cambio de venta de Bebidas. Se consulta la **Semantic View** con SQL estándar: las mismas métricas que usará el agente.

In [ ]:
%%sql -r mapa_ciudades
WITH s AS (
  SELECT * FROM SEMANTIC_VIEW(TEC_DEMO.ANALITICA.SV_COMERCIAL
    METRICS geo.latitud_centro, geo.longitud_centro, geo.num_tiendas, geo.venta_bebidas_tienda
    DIMENSIONS geo.ciudad, geo.geo_estado, geo.geo_region, geo.geo_mes
    WHERE geo.geo_mes IN ('2026-09-01', '2026-10-01')))
SELECT ciudad, geo_estado AS estado, geo_region AS region,
       AVG(latitud_centro) AS latitud, AVG(longitud_centro) AS longitud, MAX(num_tiendas) AS tiendas,
       ROUND(100 * (SUM(IFF(geo_mes = '2026-10-01', venta_bebidas_tienda, 0))
                  / NULLIF(SUM(IFF(geo_mes = '2026-09-01', venta_bebidas_tienda, 0)), 0) - 1), 1) AS cambio_bebidas_pct
FROM s GROUP BY ciudad, geo_estado, geo_region
ORDER BY cambio_bebidas_pct;

In [ ]:
df = a_pandas(mapa_ciudades, ['latitud', 'longitud', 'tiendas', 'cambio_bebidas_pct'])
cedis_df = a_pandas(cedis, ['latitud', 'longitud'])

fig = mapa_base('Cambio de venta de Bebidas sep → oct 2026 por ciudad (%)')
fig.add_trace(go.Scatter(
    x=df['longitud'], y=df['latitud'], mode='markers', name='Ciudades',
    marker={'size': (df['tiendas'] ** 0.5) * 1.6 + 5, 'color': df['cambio_bebidas_pct'], 'colorscale': 'RdYlGn',
            'cmin': -45, 'cmax': 15, 'line': {'width': 0.5, 'color': 'white'}, 'opacity': 0.9,
            'colorbar': {'title': '% cambio', 'x': 1.0}},
    text=df['ciudad'], customdata=df[['estado', 'region', 'tiendas', 'cambio_bebidas_pct']],
    hovertemplate='<b>%{text}</b>, %{customdata[0]} (%{customdata[1]})<br>%{customdata[2]:.0f} tiendas<br>Bebidas: %{customdata[3]:.1f}%<extra></extra>',
    showlegend=False))
fig.add_trace(go.Scatter(x=cedis_df['longitud'], y=cedis_df['latitud'], mode='markers+text', text=cedis_df['nombre'],
                         textposition='top right', marker={'size': 11, 'color': 'black', 'symbol': 'square'}, name='CEDIS'))
fig.show()

**Lectura:** las 24 ciudades del Bajío caen entre **−20% y −45%** alrededor del **CEDIS León**; en las otras 69 ciudades del país el cambio va de −14% a +35%. La caída tiene geografía: algo pasó en la cadena que surte desde León. (Lo confirmaremos con proveedores en 04 y con el agente en 06.)

## Mapa 2 · Densidad de venta en hexágonos H3
Snowflake convierte cada tienda a un hexágono H3 (`H3_LATLNG_TO_CELL_STRING`, resolución 7 ≈ 5 km²) y cada hexágono a su polígono (`H3_CELL_TO_BOUNDARY`). Todo en SQL, sin librerías externas.

In [ ]:
%%sql -r mapa_hex
WITH h AS (
  SELECT * FROM SEMANTIC_VIEW(TEC_DEMO.ANALITICA.SV_COMERCIAL
    METRICS geo.venta_tienda, geo.num_tiendas
    DIMENSIONS geo.h3_cell, geo.geo_region
    WHERE geo.geo_mes = '2026-10-01'))
SELECT h3_cell, geo_region AS region, ROUND(venta_tienda / 1e6, 2) AS venta_mdp, num_tiendas AS tiendas,
       ST_ASGEOJSON(H3_CELL_TO_BOUNDARY(h3_cell))::STRING AS geojson
FROM h ORDER BY venta_mdp DESC LIMIT 2000;

In [ ]:
hx = a_pandas(mapa_hex, ['venta_mdp', 'tiendas'])
hx[['lon', 'lat']] = [centro_hex(g) for g in hx['geojson']]

# A escala nacional cada hexágono H3 (res. 7, ~2.6 km) es apenas un punto: se dibuja como marcador hexagonal en su centro
fig = mapa_base('Venta de octubre 2026 por hexágono H3 (millones de pesos)')
fig.add_trace(go.Scatter(
    x=hx['lon'], y=hx['lat'], mode='markers', showlegend=False,
    marker={'symbol': 'hexagon', 'size': 6 + 14 * hx['venta_mdp'] / hx['venta_mdp'].max(), 'color': hx['venta_mdp'],
            'colorscale': 'Viridis', 'line': {'width': 0}, 'colorbar': {'title': 'MDP'}},
    customdata=hx[['h3_cell', 'region', 'tiendas', 'venta_mdp']],
    hovertemplate='Hexágono %{customdata[0]}<br>%{customdata[1]} · %{customdata[2]:.0f} tiendas<br>$%{customdata[3]:.2f} M<extra></extra>'))
fig.show()

## Mapa 3 · La frontera del problema (hexágonos, centro del país)
Color = cambio de Bebidas sep → oct por hexágono H3. Una tienda sola tiene pocas ventas al mes y su cambio es ruidoso; el hexágono suma decenas de tiendas y muestra la señal. Fíjate en la **frontera**: los hexágonos del Bajío (alrededor del CEDIS León) en rojo junto a los del Centro (CDMX, Puebla, Toluca) en verde, a pocos kilómetros.

In [ ]:
%%sql -r mapa_hex_cambio
WITH h AS (
  SELECT * FROM SEMANTIC_VIEW(TEC_DEMO.ANALITICA.SV_COMERCIAL
    METRICS geo.venta_bebidas_tienda, geo.num_tiendas
    DIMENSIONS geo.h3_cell, geo.geo_region, geo.geo_mes
    WHERE geo.geo_mes IN ('2026-09-01', '2026-10-01') AND geo.geo_region IN ('Bajío', 'Centro')))
SELECT h3_cell, geo_region AS region, MAX(num_tiendas) AS tiendas,
       ROUND(100 * (SUM(IFF(geo_mes = '2026-10-01', venta_bebidas_tienda, 0))
                  / NULLIF(SUM(IFF(geo_mes = '2026-09-01', venta_bebidas_tienda, 0)), 0) - 1), 1) AS cambio_bebidas_pct,
       ST_ASGEOJSON(H3_CELL_TO_BOUNDARY(h3_cell))::STRING AS geojson
FROM h GROUP BY h3_cell, geo_region
HAVING MAX(num_tiendas) >= 10;

In [ ]:
from plotly.colors import sample_colorscale

hc = a_pandas(mapa_hex_cambio, ['tiendas', 'cambio_bebidas_pct'])
centro = cedis_df[cedis_df['region'].isin(['Bajío', 'Centro'])]

# Acercamiento al centro del país: aquí sí se dibuja el polígono real de cada hexágono (H3_CELL_TO_BOUNDARY)
fig = mapa_base('Cambio de Bebidas sep → oct 2026 por hexágono (Bajío vs Centro)',
                lon=(-104.2, -97.0), lat=(18.4, 22.6), height=620, mares=False)
norm = ((hc['cambio_bebidas_pct'].clip(-45, 15) + 45) / 60).tolist()
colores = sample_colorscale('RdYlGn', norm)
for r, color in zip(hc.itertuples(), colores):
    pts = json.loads(r.geojson)['coordinates'][0]
    fig.add_trace(go.Scatter(x=[p[0] for p in pts], y=[p[1] for p in pts], fill='toself', mode='lines',
                             fillcolor=color, line={'color': 'white', 'width': 0.4}, opacity=0.9, showlegend=False,
                             hoverinfo='text', text=f"{r.region} · {r.tiendas:.0f} tiendas · Bebidas {r.cambio_bebidas_pct:.1f}%"))
# Barra de color (traza invisible solo para la escala)
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers', showlegend=False, hoverinfo='skip',
                         marker={'colorscale': 'RdYlGn', 'cmin': -45, 'cmax': 15, 'color': [0],
                                 'colorbar': {'title': '% cambio'}}))
fig.add_trace(go.Scatter(x=centro['longitud'], y=centro['latitud'], mode='markers+text', text=centro['nombre'],
                         textposition='top right', marker={'size': 14, 'color': 'black', 'symbol': 'square'}, name='CEDIS'))
fig.show()
print(hc.groupby('region')['cambio_bebidas_pct'].median().round(1).to_string())

## Mapa 4 · El mismo mapa como GERENTE_NORTE
No hay que programar nada nuevo: la Row Access Policy de 03 ya está en `TIENDAS_GEO`. Simulamos el rol con `EXECUTE USING POLICY_CONTEXT`.

In [ ]:
%%sql -r geo_por_rol
EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => 'GERENTE_NORTE',
    SNOWFLAKE$SESSION_ACTIVATED_ROLES => ('GERENTE_NORTE'))
  AS SELECT region, COUNT(DISTINCT tienda_id) AS tiendas_visibles, ROUND(AVG(latitud), 2) AS lat_media
     FROM TEC_DEMO.ANALITICA.TIENDAS_GEO WHERE mes = '2026-10-01' GROUP BY region ORDER BY region;

**Lectura:** en la frontera, el hexágono típico del Bajío cae **−30.8%** y el del Centro sube **+6.1%**. Y GERENTE_NORTE ve **solo** sus 4,295 tiendas del Norte: si le pide al agente "muéstrame las tiendas del Bajío en un mapa", el mapa sale vacío. La regla vive en los datos, no en el agente.

### Opcional · Competidores reales cerca de las tiendas (Overture Maps)
Con el listing gratuito **Overture Maps – Places** (CARTO, Marketplace) puedes buscar supermercados reales a ≤ 1 km de las tiendas que más cayeron. Instálalo como `OVERTURE_MAPS__PLACES` y descomenta la celda siguiente. Los resultados son datos públicos de referencia, no una afirmación sobre esos negocios.

In [ ]:
%%sql -r competidores_cerca
-- Requiere el listing Overture Maps - Places (CARTO). Descomentar después de instalarlo.
-- WITH peores AS (
--   SELECT tienda_id, tienda, latitud, longitud FROM TEC_DEMO.ANALITICA.TIENDAS_GEO
--   WHERE region = 'Bajío' AND ciudad = 'León' AND mes = '2026-10-01' ORDER BY venta_bebidas LIMIT 20)
-- SELECT p.tienda, pl.names:primary::STRING AS competidor, pl.categories:primary::STRING AS categoria,
--        ST_Y(pl.geometry) AS latitud, ST_X(pl.geometry) AS longitud,
--        ROUND(ST_DISTANCE(pl.geometry, ST_MAKEPOINT(p.longitud, p.latitud))) AS distancia_m
-- FROM peores p JOIN OVERTURE_MAPS__PLACES.CARTO.PLACE pl
--   ON ST_DWITHIN(pl.geometry, ST_MAKEPOINT(p.longitud, p.latitud), 1000)
-- WHERE pl.categories:primary::STRING IN ('supermarket', 'convenience_store', 'grocery_store')
-- ORDER BY p.tienda, distancia_m LIMIT 200;
SELECT 'Opcional: instala Overture Maps - Places y descomenta' AS nota;